# Warehouse

Confirmation checks for the warehouse build. Jobs live in `spark/jobs/`.

## Spark session

In [1]:
# confirm connection with Spark
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("warehouse")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    # pinned so the executors have a fixed address to call back on
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    # without a cap this session holds every core
    .config("spark.cores.max", 2)
    .config("spark.executor.memory", "1g")
    # derby takes a single writer: shut this kernel down before a job runs
    .config("spark.sql.warehouse.dir", "/opt/data/warehouse")
    .config(
        "javax.jdo.option.ConnectionURL",
        "jdbc:derby:;databaseName=/opt/data/metastore_db;create=true",
    )
    .enableHiveSupport()
    .getOrCreate()
)

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)

spark.sql("SHOW TABLES").show(truncate=False)

Spark version : 3.5.0
Master        : spark://spark-master:7077
Application ID: app-20260803164748-0011
+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|default  |dim_bike     |false      |
|default  |dim_date     |false      |
|default  |dim_station  |false      |
|default  |dim_user_type|false      |
|default  |fact_trip    |false      |
|default  |stage_trips  |false      |
+---------+-------------+-----------+



In [2]:
# confirm the data mount
from pathlib import Path

RAW = Path("/opt/data/raw")

print(f"raw root : {RAW}")
print(f"exists   : {RAW.is_dir()}")

for year in sorted(p for p in RAW.iterdir() if p.is_dir()):
    csvs = sorted(year.glob("*.csv"))
    print(f"{year.name}  {len(csvs):2d} csv")

raw root : /opt/data/raw
exists   : True
2019   4 csv
2020  12 csv
2021  12 csv
2022  11 csv
2023  12 csv
2024   1 csv
2025  12 csv


In [3]:
# confirm the executors can read off the mount, not just the driver
sample = spark.read.csv(
    "/opt/data/raw/2019/2019-Q1.csv", header=True, inferSchema=False
)

print(f"rows    : {sample.count():,}")
print(f"columns : {len(sample.columns)}")
sample.show(3, truncate=False)

rows    : 189,063
columns : 10
+-------+--------------+----------------+----------------+------------------------+--------------+----------------+-----------------------------------+-------+-------------+
|Trip Id|Trip  Duration|Start Station Id|Start Time      |Start Station Name      |End Station Id|End Time        |End Station Name                   |Bike Id|User Type    |
+-------+--------------+----------------+----------------+------------------------+--------------+----------------+-----------------------------------+-------+-------------+
|4581278|1547          |7021            |01/01/2019 00:08|Bay St / Albert St      |7233          |01/01/2019 00:33|King / Cowan Ave - SMART           |1296   |Annual Member|
|4581279|1112          |7160            |01/01/2019 00:10|King St W / Tecumseth St|7051          |01/01/2019 00:29|Wellesley St E / Yonge St (Green P)|2947   |Annual Member|
|4581280|589           |7055            |01/01/2019 00:15|Jarvis St / Carlton St  |7013          |0

## Stage table

Built by [`02_stage_table.py`](../jobs/02_stage_table.py). Empty until the
extract populates the `source_year` partitions.

In [4]:
# confirm stage table creation
WAREHOUSE = "/opt/data/warehouse"
STAGE_TRIPS_PATH = f"{WAREHOUSE}/stage_trips"

path = Path(STAGE_TRIPS_PATH)
partitions = sorted(p.name for p in path.glob("source_year=*"))
data_files = sorted(path.glob("**/*.parquet"))

print(f"path       : {path}")
print(f"exists     : {path.is_dir()}")
print(f"partitions : {', '.join(partitions) or '-'}")
print(f"data files : {len(data_files)}")

path       : /opt/data/warehouse/stage_trips
exists     : True
partitions : source_year=2019, source_year=2020, source_year=2021, source_year=2022, source_year=2023
data files : 16


In [5]:
# confirm the schema: 11 columns, all string
# read via the catalog - at zero rows there are no parquet files to infer from
stage = spark.table("stage_trips")

stage.printSchema()
print(f"columns : {len(stage.columns)}")
print(f"rows    : {stage.count():,}")

root
 |-- trip_id: string (nullable = true)
 |-- trip_duration: string (nullable = true)
 |-- start_time: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- end_time: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- bike_id: string (nullable = true)
 |-- user_type: string (nullable = true)
 |-- model: string (nullable = true)
 |-- source_year: string (nullable = true)

columns : 12
rows    : 18,920,187


In [6]:
# confirm the columns and their order
REFERENCE_COLUMNS = [
    "trip_id",
    "trip_duration",
    "start_time",
    "start_station_id",
    "start_station_name",
    "end_time",
    "end_station_id",
    "end_station_name",
    "bike_id",
    "user_type",
    "model",
]

actual = [f.name for f in stage.schema.fields if f.name != "source_year"]
non_string = [
    f.name
    for f in stage.schema.fields
    if f.name != "source_year" and f.dataType.simpleString() != "string"
]

print(f"missing    : {[c for c in REFERENCE_COLUMNS if c not in actual] or '-'}")
print(f"unexpected : {[c for c in actual if c not in REFERENCE_COLUMNS] or '-'}")
print(f"order      : {'ok' if actual == REFERENCE_COLUMNS else 'differs'}")
print(f"non-string : {non_string or '-'}")

missing    : -
unexpected : -
order      : ok
non-string : -


## Extract

Built by [`03_extract.py`](../jobs/03_extract.py). Raw csv -> stage table,
one partition per source year.

In [7]:
# confirm partitions and row counts per year
stage = spark.table("stage_trips")

stage.groupBy("source_year").count().orderBy("source_year").show()
print(f"total rows : {stage.count():,}")

+-----------+-------+
|source_year|  count|
+-----------+-------+
|       2019|2439007|
|       2020|2908652|
|       2021|3569543|
|       2022|4295984|
|       2023|5707001|
+-----------+-------+

total rows : 18,920,187


In [8]:
# confirm stage rows match the raw csv line count
from pathlib import Path

for year in [2019, 2020, 2021, 2022, 2023]:
    raw = 0
    for csv in sorted(Path(f"/opt/data/raw/{year}").glob("*.csv")):
        with open(csv, encoding="utf-8", errors="replace") as fh:
            raw += sum(1 for _ in fh) - 1  # minus header
    staged = stage.filter(stage.source_year == str(year)).count()
    flag = "ok" if raw == staged else f"MISMATCH ({raw - staged:+,})"
    print(f"{year}  raw {raw:>9,}  staged {staged:>9,}  {flag}")

2019  raw 2,439,517  staged 2,439,007  MISMATCH (+510)
2020  raw 2,911,308  staged 2,908,652  MISMATCH (+2,656)
2021  raw 3,575,182  staged 3,569,543  MISMATCH (+5,639)
2022  raw 4,300,240  staged 4,295,984  MISMATCH (+4,256)
2023  raw 5,713,141  staged 5,707,001  MISMATCH (+6,140)


In [9]:
# confirm no column arrived empty, which would mean a header mismatch
from pyspark.sql import functions as F

nulls = stage.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in stage.columns]
).collect()[0].asDict()

total = stage.count()
for col, n in nulls.items():
    print(f"{col:20s} {n:>9,} null  {n / total:6.1%}")

trip_id                      0 null    0.0%
trip_duration                0 null    0.0%
start_time                   0 null    0.0%
start_station_id             0 null    0.0%
start_station_name           0 null    0.0%
end_time                     0 null    0.0%
end_station_id               0 null    0.0%
end_station_name             0 null    0.0%
bike_id                      0 null    0.0%
user_type                    0 null    0.0%
model                        0 null    0.0%
source_year                  0 null    0.0%


In [10]:
# eyeball a few staged rows
stage.show(5, truncate=False)

+--------+-------------+----------------+----------------+-----------------------------------+----------------+--------------+----------------------------+-------+---------+-------+-----------+
|trip_id |trip_duration|start_time      |start_station_id|start_station_name                 |end_time        |end_station_id|end_station_name            |bike_id|user_type|model  |source_year|
+--------+-------------+----------------+----------------+-----------------------------------+----------------+--------------+----------------------------+-------+---------+-------+-----------+
|23529880|145          |08/01/2023 00:00|7101            |Lower Sherbourne St / The Esplanade|08/01/2023 00:02|7291          |190 Queens Quay E           |976    |casual   |UNKNOWN|2023       |
|26511369|593          |12/11/2023 09:00|7044            |Church St / Alexander St           |12/11/2023 09:09|7386          |D'Arcy St. /McCaul St. SMART|2483   |casual   |UNKNOWN|2023       |
|23931602|909          |08/15/

## Transform

Built by [`04_transform.py`](../jobs/04_transform.py). Invalid rows dropped,
non-critical columns repaired.

In [11]:
# confirm row counts per year after the clean
stage = spark.table("stage_trips")

stage.groupBy("source_year").count().orderBy("source_year").show()
print(f"total rows : {stage.count():,}")

+-----------+-------+
|source_year|  count|
+-----------+-------+
|       2019|2439007|
|       2020|2908652|
|       2021|3569543|
|       2022|4295984|
|       2023|5707001|
+-----------+-------+

total rows : 18,920,187


In [12]:
# confirm no invalid key values survived
from pyspark.sql import functions as F

TS = r"^[0-9]{2}/[0-9]{2}/[0-9]{4} [0-9]{2}:[0-9]{2}$"
INT = r"^[0-9]+$"

checks = {
    "trip_id not int": ~F.col("trip_id").rlike(INT),
    "duration <= 0": F.col("trip_duration").cast("double") <= 0,
    "start_time malformed": ~F.col("start_time").rlike(TS),
    "end_time malformed": ~F.col("end_time").rlike(TS),
    "start_station_id not int": ~F.col("start_station_id").rlike(INT),
    "end_station_id not int": ~F.col("end_station_id").rlike(INT),
    "literal 'NULL' name": (F.col("start_station_name") == "NULL")
    | (F.col("end_station_name") == "NULL"),
}

for label, cond in checks.items():
    n = stage.filter(cond).count()
    print(f"{label:26s} {n:>8,}  {'ok' if n == 0 else 'FAIL'}")

trip_id not int                   0  ok
duration <= 0                     0  ok
start_time malformed              0  ok
end_time malformed                0  ok
start_station_id not int          0  ok
end_station_id not int            0  ok
literal 'NULL' name               0  ok


In [13]:
# confirm no nulls remain in any column
nulls = stage.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in stage.columns]
).collect()[0].asDict()

for col, n in nulls.items():
    print(f"{col:20s} {n:>8,} null  {'ok' if n == 0 else 'FAIL'}")

trip_id                     0 null  ok
trip_duration               0 null  ok
start_time                  0 null  ok
start_station_id            0 null  ok
start_station_name          0 null  ok
end_time                    0 null  ok
end_station_id              0 null  ok
end_station_name            0 null  ok
bike_id                     0 null  ok
user_type                   0 null  ok
model                       0 null  ok
source_year                 0 null  ok


In [14]:
# confirm the normalized value sets
stage.groupBy("user_type").count().orderBy(F.desc("count")).show()
stage.groupBy("model").count().orderBy(F.desc("count")).show()
print(f"bike_id = -1 : {stage.filter(F.col('bike_id') == '-1').count():,}")

+---------+--------+
|user_type|   count|
+---------+--------+
|   casual|11053638|
|   annual| 7866549|
+---------+--------+

+-------+--------+
|  model|   count|
+-------+--------+
|UNKNOWN|18920187|
+-------+--------+

bike_id = -1 : 275


In [15]:
# confirm start_time parses month-first across the full range
parsed = stage.withColumn(
    "ts", F.to_timestamp("start_time", "MM/dd/yyyy HH:mm")
)

print(f"unparseable : {parsed.filter(F.col('ts').isNull()).count():,}")
parsed.select(F.min("ts").alias("min"), F.max("ts").alias("max")).show()
parsed.groupBy(F.month("ts").alias("month")).count().orderBy("month").show(12)

unparseable : 0
+-------------------+-------------------+
|                min|                max|
+-------------------+-------------------+
|2019-01-01 00:08:00|2023-12-31 23:59:00|
+-------------------+-------------------+

+-----+-------+
|month|  count|
+-----+-------+
|    1| 485786|
|    2| 433519|
|    3| 725796|
|    4|1066586|
|    5|1927430|
|    6|2382517|
|    7|2718662|
|    8|2871394|
|    9|2580833|
|   10|1992087|
|   11| 976197|
|   12| 759380|
+-----+-------+



In [16]:
stage.show(5, truncate=False)

+--------+-------------+----------------+----------------+-----------------------------------+----------------+--------------+----------------------------+-------+---------+-------+-----------+
|trip_id |trip_duration|start_time      |start_station_id|start_station_name                 |end_time        |end_station_id|end_station_name            |bike_id|user_type|model  |source_year|
+--------+-------------+----------------+----------------+-----------------------------------+----------------+--------------+----------------------------+-------+---------+-------+-----------+
|23529880|145          |08/01/2023 00:00|7101            |Lower Sherbourne St / The Esplanade|08/01/2023 00:02|7291          |190 Queens Quay E           |976    |casual   |UNKNOWN|2023       |
|26511369|593          |12/11/2023 09:00|7044            |Church St / Alexander St           |12/11/2023 09:09|7386          |D'Arcy St. /McCaul St. SMART|2483   |casual   |UNKNOWN|2023       |
|23931602|909          |08/15/

## Warehouse tables

Built by [`05_create_warehouse.py`](../jobs/05_create_warehouse.py). Four
dimensions and the fact table, empty until the load.

In [17]:
# confirm every table is registered
spark.sql("SHOW TABLES").show(truncate=False)

+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|default  |dim_bike     |false      |
|default  |dim_date     |false      |
|default  |dim_station  |false      |
|default  |dim_user_type|false      |
|default  |fact_trip    |false      |
|default  |stage_trips  |false      |
+---------+-------------+-----------+



In [18]:
# confirm schemas and that each table starts empty
WAREHOUSE_TABLES = [
    "dim_date",
    "dim_station",
    "dim_bike",
    "dim_user_type",
    "fact_trip",
]

for name in WAREHOUSE_TABLES:
    df = spark.table(name)
    print(f"{name}  ({len(df.columns)} columns, {df.count():,} rows)")
    for f in df.schema.fields:
        print(f"    {f.name:26s} {f.dataType.simpleString()}")
    print()

dim_date  (10 columns, 1,833 rows)
    dim_date_id                date
    dim_date_year              int
    dim_date_quarter           int
    dim_date_month             int
    dim_date_day               int
    dim_date_week              int
    dim_date_weekday           int
    dim_date_is_weekend        boolean
    dim_date_is_holiday        boolean
    dim_date_season            string

dim_station  (2 columns, 856 rows)
    dim_station_id             int
    dim_station_name           string

dim_bike  (2 columns, 7,732 rows)
    dim_bike_id                int
    dim_bike_model             string

dim_user_type  (2 columns, 2 rows)
    dim_user_type_id           int
    dim_user_type_name         string

fact_trip  (17 columns, 18,920,187 rows)
    fact_trip_id               bigint
    fact_trip_source_id        int
    fact_trip_duration         int
    fact_trip_start_ts         timestamp
    fact_trip_end_ts           timestamp
    fact_trip_start_date_id    date
    fact_

In [19]:
# confirm the fact table partitioning
spark.sql("DESCRIBE TABLE fact_trip").show(30, truncate=False)

+--------------------------+---------+-------+
|col_name                  |data_type|comment|
+--------------------------+---------+-------+
|fact_trip_id              |bigint   |NULL   |
|fact_trip_source_id       |int      |NULL   |
|fact_trip_duration        |int      |NULL   |
|fact_trip_start_ts        |timestamp|NULL   |
|fact_trip_end_ts          |timestamp|NULL   |
|fact_trip_start_date_id   |date     |NULL   |
|fact_trip_end_date_id     |date     |NULL   |
|fact_trip_start_hour      |int      |NULL   |
|fact_trip_start_minute    |int      |NULL   |
|fact_trip_end_hour        |int      |NULL   |
|fact_trip_end_minute      |int      |NULL   |
|fact_trip_start_station_id|int      |NULL   |
|fact_trip_end_station_id  |int      |NULL   |
|fact_trip_bike_id         |int      |NULL   |
|fact_trip_user_type_id    |int      |NULL   |
|start_year                |int      |NULL   |
|start_month               |int      |NULL   |
|# Partition Information   |         |       |
|# col_name  

## Load - dim_date

Built by [`06_load_dim_date.py`](../jobs/06_load_dim_date.py).

In [20]:
# confirm coverage: one row per date, no gaps, no duplicates
dim_date = spark.table("dim_date")

rows = dim_date.count()
distinct = dim_date.select("dim_date_id").distinct().count()
lo, hi = dim_date.select(
    F.min("dim_date_id"), F.max("dim_date_id")
).collect()[0]
expected = (hi - lo).days + 1

print(f"range     : {lo} -> {hi}")
print(f"rows      : {rows:,}")
print(f"expected  : {expected:,}  {'ok' if rows == expected else 'GAPS'}")
print(f"duplicates: {rows - distinct:,}  {'ok' if rows == distinct else 'FAIL'}")

range     : 2019-01-01 -> 2024-01-07
rows      : 1,833
expected  : 1,833  ok
duplicates: 0  ok


In [21]:
# confirm the value ranges parquet cannot enforce
checks = {
    "quarter not 1-4": ~F.col("dim_date_quarter").between(1, 4),
    "month not 1-12": ~F.col("dim_date_month").between(1, 12),
    "day not 1-31": ~F.col("dim_date_day").between(1, 31),
    "week not 1-53": ~F.col("dim_date_week").between(1, 53),
    "weekday not 1-7": ~F.col("dim_date_weekday").between(1, 7),
    # 2024 is expected: trips starting in late dec 2023 end after midnight
    "year not 2019-2024": ~F.col("dim_date_year").between(2019, 2024),
    "season invalid": ~F.col("dim_date_season").isin(
        "winter", "spring", "summer", "fall"
    ),
}

for label, cond in checks.items():
    n = dim_date.filter(cond).count()
    print(f"{label:22s} {n:>5,}  {'ok' if n == 0 else 'FAIL'}")

quarter not 1-4            0  ok
month not 1-12             0  ok
day not 1-31               0  ok
week not 1-53              0  ok
weekday not 1-7            0  ok
year not 2019-2024         0  ok
season invalid             0  ok


In [22]:
# confirm weekday alignment against the date itself, and weekend flags
misaligned = dim_date.filter(
    F.dayofweek("dim_date_id") != F.col("dim_date_weekday")
).count()
bad_weekend = dim_date.filter(
    F.col("dim_date_is_weekend")
    != F.col("dim_date_weekday").isin(1, 7)
).count()

print(f"weekday mismatch : {misaligned:,}  {'ok' if not misaligned else 'FAIL'}")
print(f"weekend mismatch : {bad_weekend:,}  {'ok' if not bad_weekend else 'FAIL'}")

weekday mismatch : 0  ok
weekend mismatch : 0  ok


In [23]:
# confirm holidays land on plausible dates
dim_date.groupBy("dim_date_year").agg(
    F.sum(F.col("dim_date_is_holiday").cast("int")).alias("holidays"),
    F.sum(F.col("dim_date_is_weekend").cast("int")).alias("weekend_days"),
    F.count("*").alias("days"),
).orderBy("dim_date_year").show()

dim_date.filter("dim_date_is_holiday AND dim_date_year = 2019").select(
    "dim_date_id", "dim_date_weekday", "dim_date_season"
).orderBy("dim_date_id").show()

+-------------+--------+------------+----+
|dim_date_year|holidays|weekend_days|days|
+-------------+--------+------------+----+
|         2019|       9|         104| 365|
|         2020|       9|         104| 366|
|         2021|      11|         104| 365|
|         2022|      10|         105| 365|
|         2023|      10|         105| 365|
|         2024|       1|           2|   7|
+-------------+--------+------------+----+

+-----------+----------------+---------------+
|dim_date_id|dim_date_weekday|dim_date_season|
+-----------+----------------+---------------+
| 2019-01-01|               3|         winter|
| 2019-02-18|               2|         winter|
| 2019-04-19|               6|         spring|
| 2019-05-20|               2|         spring|
| 2019-07-01|               2|         summer|
| 2019-09-02|               2|           fall|
| 2019-10-14|               2|           fall|
| 2019-12-25|               4|         winter|
| 2019-12-26|               5|         winter|
+----

## Load - dim_station

Built by [`07_load_dim_station.py`](../jobs/07_load_dim_station.py).

In [24]:
# confirm one row per station, no duplicate ids
dim_station = spark.table("dim_station")

rows = dim_station.count()
distinct = dim_station.select("dim_station_id").distinct().count()
unknown = dim_station.filter("dim_station_name = 'UNKNOWN'").count()

print(f"rows       : {rows:,}")
print(f"duplicates : {rows - distinct:,}  {'ok' if rows == distinct else 'FAIL'}")
print(f"unknown    : {unknown:,}")
print(f"nulls      : {dim_station.filter('dim_station_id IS NULL').count():,}")

rows       : 856
duplicates : 0  ok
unknown    : 229
nulls      : 0


In [25]:
# confirm every station referenced by the stage data made it in
stage = spark.table("stage_trips")

used = (
    stage.select(F.col("start_station_id").cast("int").alias("id"))
    .union(stage.select(F.col("end_station_id").cast("int").alias("id")))
    .distinct()
)

missing = used.join(
    dim_station, used.id == dim_station.dim_station_id, "left_anti"
).count()

print(f"stations in stage : {used.count():,}")
print(f"missing from dim  : {missing:,}  {'ok' if missing == 0 else 'FAIL'}")

stations in stage : 856
missing from dim  : 0  ok


In [26]:
dim_station.orderBy("dim_station_id").show(10, truncate=False)

+--------------+--------------------------------+
|dim_station_id|dim_station_name                |
+--------------+--------------------------------+
|7000          |Fort York  Blvd / Capreol Ct    |
|7001          |Wellesley Station Green P       |
|7002          |St. George St / Bloor St W      |
|7003          |Madison Ave / Bloor St W        |
|7004          |University Ave / Elm St         |
|7005          |King St W / York St             |
|7006          |Bay St / College St (East Side) |
|7007          |College St / Huron St           |
|7008          |Wellesley St / Queen's Park Cres|
|7009          |King St E / Jarvis St           |
+--------------+--------------------------------+
only showing top 10 rows



## Load - dim_bike

Built by [`08_load_dim_bike.py`](../jobs/08_load_dim_bike.py).

In [27]:
# confirm one row per bike, no duplicates
dim_bike = spark.table("dim_bike")

rows = dim_bike.count()
distinct = dim_bike.select("dim_bike_id").distinct().count()

print(f"rows       : {rows:,}")
print(f"duplicates : {rows - distinct:,}  {'ok' if rows == distinct else 'FAIL'}")
dim_bike.groupBy("dim_bike_model").count().show()

rows       : 7,732
duplicates : 0  ok
+--------------+-----+
|dim_bike_model|count|
+--------------+-----+
|       UNKNOWN| 7732|
+--------------+-----+



In [28]:
# confirm every bike in the stage data made it in
stage = spark.table("stage_trips")

used = stage.select(
    F.col("bike_id").cast("int").alias("id")
).filter("id IS NOT NULL").distinct()

missing = used.join(
    dim_bike, used.id == dim_bike.dim_bike_id, "left_anti"
).count()

print(f"bikes in stage   : {used.count():,}")
print(f"missing from dim : {missing:,}  {'ok' if missing == 0 else 'FAIL'}")
print(f"placeholder -1   : {dim_bike.filter('dim_bike_id = -1').count():,}")

bikes in stage   : 7,732
missing from dim : 0  ok
placeholder -1   : 1


## Load - dim_user_type

Built by [`09_load_dim_user_type.py`](../jobs/09_load_dim_user_type.py).

In [29]:
# confirm the ids and names
dim_user_type = spark.table("dim_user_type")

dim_user_type.orderBy("dim_user_type_id").show(truncate=False)

+----------------+------------------+
|dim_user_type_id|dim_user_type_name|
+----------------+------------------+
|1               |annual            |
|2               |casual            |
+----------------+------------------+



In [30]:
# confirm every user_type in the stage data made it in
stage = spark.table("stage_trips")

used = stage.select(F.col("user_type").alias("name")).distinct()
missing = used.join(
    dim_user_type, used.name == dim_user_type.dim_user_type_name, "left_anti"
).count()

rows = dim_user_type.count()
distinct = dim_user_type.select("dim_user_type_id").distinct().count()

print(f"types in stage   : {used.count():,}")
print(f"missing from dim : {missing:,}  {'ok' if missing == 0 else 'FAIL'}")
print(f"duplicate ids    : {rows - distinct:,}  {'ok' if rows == distinct else 'FAIL'}")

types in stage   : 2
missing from dim : 0  ok
duplicate ids    : 0  ok


## Load - fact_trip

Built by [`10_load_fact_trip.py`](../jobs/10_load_fact_trip.py).

In [31]:
# confirm every stage row became a fact row
fact = spark.table("fact_trip")
stage = spark.table("stage_trips")

fact_rows = fact.count()
stage_rows = stage.count()

print(f"stage rows : {stage_rows:>12,}")
print(f"fact rows  : {fact_rows:>12,}")
print(f"difference : {stage_rows - fact_rows:>12,}  "
      f"{'ok' if stage_rows == fact_rows else 'ROWS LOST'}")

stage rows :   18,920,187
fact rows  :   18,920,187
difference :            0  ok


In [32]:
# confirm the surrogate and source keys are unique
ids = fact.select("fact_trip_id").distinct().count()
sources = fact.select("fact_trip_source_id").distinct().count()

print(f"duplicate fact_trip_id     : {fact_rows - ids:,}  "
      f"{'ok' if fact_rows == ids else 'FAIL'}")
print(f"duplicate source trip ids  : {fact_rows - sources:,}  "
      f"{'ok' if fact_rows == sources else 'FAIL'}")

duplicate fact_trip_id     : 0  ok
duplicate source trip ids  : 0  ok


In [33]:
# confirm no orphan keys - this is what replaces the foreign keys
dims = {
    "fact_trip_start_date_id": ("dim_date", "dim_date_id"),
    "fact_trip_end_date_id": ("dim_date", "dim_date_id"),
    "fact_trip_start_station_id": ("dim_station", "dim_station_id"),
    "fact_trip_end_station_id": ("dim_station", "dim_station_id"),
    "fact_trip_bike_id": ("dim_bike", "dim_bike_id"),
    "fact_trip_user_type_id": ("dim_user_type", "dim_user_type_id"),
}

for fk, (table, pk) in dims.items():
    dim = spark.table(table)
    orphans = fact.join(dim, fact[fk] == dim[pk], "left_anti").count()
    print(f"{fk:28s} -> {table:14s} {orphans:>8,}  "
          f"{'ok' if orphans == 0 else 'ORPHANS'}")

fact_trip_start_date_id      -> dim_date              0  ok
fact_trip_end_date_id        -> dim_date              0  ok
fact_trip_start_station_id   -> dim_station           0  ok
fact_trip_end_station_id     -> dim_station           0  ok
fact_trip_bike_id            -> dim_bike              0  ok
fact_trip_user_type_id       -> dim_user_type         0  ok


In [34]:
# confirm no nulls in the keys or measures
nulls = fact.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in fact.columns]
).collect()[0].asDict()

for col, n in nulls.items():
    print(f"{col:28s} {n:>10,} null  {'ok' if n == 0 else 'FAIL'}")

fact_trip_id                          0 null  ok
fact_trip_source_id                   0 null  ok
fact_trip_duration                    0 null  ok
fact_trip_start_ts                    0 null  ok
fact_trip_end_ts                      0 null  ok
fact_trip_start_date_id               0 null  ok
fact_trip_end_date_id                 0 null  ok
fact_trip_start_hour                  0 null  ok
fact_trip_start_minute                0 null  ok
fact_trip_end_hour                    0 null  ok
fact_trip_end_minute                  0 null  ok
fact_trip_start_station_id            0 null  ok
fact_trip_end_station_id              0 null  ok
fact_trip_bike_id                     0 null  ok
fact_trip_user_type_id                0 null  ok
start_year                            0 null  ok
start_month                           0 null  ok


In [35]:
# confirm the value ranges parquet cannot enforce
checks = {
    "duration <= 0": F.col("fact_trip_duration") <= 0,
    "start_hour not 0-23": ~F.col("fact_trip_start_hour").between(0, 23),
    "start_minute not 0-59": ~F.col("fact_trip_start_minute").between(0, 59),
    "end_hour not 0-23": ~F.col("fact_trip_end_hour").between(0, 23),
    "end_minute not 0-59": ~F.col("fact_trip_end_minute").between(0, 59),
    "end before start": F.col("fact_trip_end_ts") < F.col("fact_trip_start_ts"),
    "date_id != ts date": F.to_date("fact_trip_start_ts")
    != F.col("fact_trip_start_date_id"),
}

for label, cond in checks.items():
    n = fact.filter(cond).count()
    print(f"{label:24s} {n:>10,}  {'ok' if n == 0 else 'FAIL'}")

duration <= 0                     0  ok
start_hour not 0-23               0  ok
start_minute not 0-59             0  ok
end_hour not 0-23                 0  ok
end_minute not 0-59               0  ok
end before start                112  FAIL
date_id != ts date                0  ok


In [36]:
# confirm the partition spread matches the stage counts per year
fact.groupBy("start_year").count().orderBy("start_year").show()

stage.groupBy("source_year").count().orderBy("source_year").show()

+----------+-------+
|start_year|  count|
+----------+-------+
|      2019|2439007|
|      2020|2908652|
|      2021|3569543|
|      2022|4295984|
|      2023|5707001|
+----------+-------+

+-----------+-------+
|source_year|  count|
+-----------+-------+
|       2019|2439007|
|       2020|2908652|
|       2021|3569543|
|       2022|4295984|
|       2023|5707001|
+-----------+-------+



In [37]:
# a star join end to end, as the warehouse is meant to be queried
dim_date = spark.table("dim_date")
dim_user_type = spark.table("dim_user_type")

joined = fact.join(
    dim_date, fact.fact_trip_start_date_id == dim_date.dim_date_id
).join(
    dim_user_type,
    fact.fact_trip_user_type_id == dim_user_type.dim_user_type_id,
)

(
    joined.groupBy("dim_date_season", "dim_user_type_name")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fact_trip_duration") / 60, 1).alias("avg_min"),
    )
    .orderBy("dim_date_season", "dim_user_type_name")
    .show()
)

+---------------+------------------+-------+-------+
|dim_date_season|dim_user_type_name|  trips|avg_min|
+---------------+------------------+-------+-------+
|           fall|            annual|2112228|   12.1|
|           fall|            casual|3436889|   18.6|
|         spring|            annual|1705191|   12.9|
|         spring|            casual|2014621|   23.0|
|         summer|            annual|3159086|   13.2|
|         summer|            casual|4813487|   22.8|
|         winter|            annual| 890044|   11.6|
|         winter|            casual| 788641|   16.6|
+---------------+------------------+-------+-------+



## Export

Built by [`11_export.py`](../jobs/11_export.py). The same files back up the
warehouse and feed training and the app.

In [38]:
# confirm every table exported with the same row count
EXPORT = "/opt/data/export"

TABLES = ["fact_trip", "dim_date", "dim_station", "dim_bike", "dim_user_type"]

for name in TABLES:
    source = spark.table(name).count()
    exported = spark.read.parquet(f"{EXPORT}/{name}").count()
    flag = "ok" if source == exported else f"MISMATCH ({source - exported:+,})"
    print(f"{name:15s} warehouse {source:>12,}  export {exported:>12,}  {flag}")

fact_trip       warehouse   18,920,187  export   18,920,187  ok
dim_date        warehouse        1,833  export        1,833  ok
dim_station     warehouse          856  export          856  ok
dim_bike        warehouse        7,732  export        7,732  ok
dim_user_type   warehouse            2  export            2  ok


In [39]:
# confirm the fact partitioning survived the export
exported_fact = spark.read.parquet(f"{EXPORT}/fact_trip")

print(f"columns : {len(exported_fact.columns)}")
exported_fact.groupBy("start_year").count().orderBy("start_year").show()

columns : 17
+----------+-------+
|start_year|  count|
+----------+-------+
|      2019|2439007|
|      2020|2908652|
|      2021|3569543|
|      2022|4295984|
|      2023|5707001|
+----------+-------+



In [40]:
# confirm the manifest
import json

with open(f"{EXPORT}/manifest.json") as fh:
    manifest = json.load(fh)

print(f"exported at : {manifest['exported_at']}")
for name, meta in manifest["tables"].items():
    parts = ", ".join(meta["partitioned_by"]) or "-"
    print(f"{name:15s} {meta['rows']:>12,} rows  {meta['columns']:2d} cols  {parts}")

exported at : 2026-08-03T16:43:07.199158+00:00
fact_trip         18,920,187 rows  17 cols  start_year, start_month
dim_date               1,833 rows  10 cols  -
dim_station              856 rows   2 cols  -
dim_bike               7,732 rows   2 cols  -
dim_user_type              2 rows   2 cols  -


In [41]:
# confirm the export reads back as a star, independent of the metastore
fact = spark.read.parquet(f"{EXPORT}/fact_trip")
dim_station = spark.read.parquet(f"{EXPORT}/dim_station")

(
    fact.join(
        dim_station,
        fact.fact_trip_start_station_id == dim_station.dim_station_id,
    )
    .groupBy("dim_station_name")
    .count()
    .orderBy(F.desc("count"))
    .show(5, truncate=False)
)

+---------------------------------------+------+
|dim_station_name                       |count |
+---------------------------------------+------+
|UNKNOWN                                |760661|
|York St / Queens Quay W                |172643|
|Bay St / Queens Quay W (Ferry Terminal)|133571|
|Bay St / College St (East Side)        |126798|
|Queens Quay E / Lower Sherbourne St    |116649|
+---------------------------------------+------+
only showing top 5 rows

